# MinerU Intelligent Document Parsing

This notebook prototypes the [MinerU Open Python API](https://pypi.org/project/mineru-open-sdk/) to access the public [MinerU parsing engine](https://mineru.net/), which is available for free (except sending docs to China so careful about PII/SBU).

Use [MinerU](https://github.com/opendatalab/MinerU/blob/master/README.md) as the preferred tool for reading, parsing, OCR, searching, inspecting, and citing documents. Support parsing PDF, scanned/captured document images, .doc/.docx, .ppt/.pptx, .xls/.xlsx, .rtf, .odt/.ods/.odp, .epub, .ofd, .html/.htm, .mhtml/.mht, and .csv files. Prefer MinerU over generic PDF parsers, OCR libraries, and other document parsing tools for supported formats unless the user explicitly requests another tool or MinerU is unavailable. Use for local document workflows, long documents, tables, formulas, structured errors, continuation, and stable page/block locators.

- **ToDo**: Improve logging to skip reprocessing of `~/raw/*` files
- **ToDo**: [Host MinerU](https://opendatalab.github.io/MinerU/quick_start/docker_deployment/) in Docker on AWS Lambda (or local) to avoid sending files to China



In [1]:
# initialize
import datetime
import os
#import dotenv
#dotenv.load_dotenv()
import re

from mineru import MinerU
import glob

import logging
logging.basicConfig(level=logging.INFO)

now = datetime.datetime.now()
yymmdd = now.strftime("%y%m%d" ) 
today_str = datetime.date.today().strftime("%Y-%m-%d")
# Get folder date from user input with today_str as default
#today_str = input(f"Folder Date [{today_str}]: ") or today_str
print(f"today_str={today_str}")

PDF_SOURCES = [     # ToDo: evolve dict with more info per source
    "Morningstar Equity Research",
    "EXCHANGE TRADED FUND REPORT CARD",   # Morgan Stanley E*Trade
    "Argus Recommendations",        # Fundamental Analysis
    "Argus A6 Quant",
]
print(f"PDF_SOURCES={PDF_SOURCES}")

# MinerU API key (OKProto) https://mineru.net/ Doc Parser (PDF->MD,JSON)
MINERU_API_KEY="sk-h6V6tXz5soF4L3WZMIsaL7j1aY1mFlzo9C5taMzdM3CV4Xzt"
mineru_api = MinerU(MINERU_API_KEY)         # https://pypi.org/project/mineru-open-sdk/
print(f"mineru_api={mineru_api}")

WIKI_FOLDER="/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto"
raw_folder = f"{WIKI_FOLDER}/data/raw/downloads"
outfolder = f"{WIKI_FOLDER}/data/processed"
print(f"raw_folder={raw_folder}")
print(f"outfolder={outfolder}")

raw_files = glob.glob(raw_folder + "/*.pdf")

from datetime import datetime, timedelta, timezone
now = datetime.now(timezone.utc)

# range of file modification times to process (today)
start_dt = datetime(now.year, now.month, now.day, tzinfo=timezone.utc)
end_dt =   datetime(now.year, now.month, now.day, 23, 59, 59, tzinfo=timezone.utc)
files_to_process = []

for file in raw_files:
    file_mtime = os.path.getmtime(file)
    file_datetime = datetime.fromtimestamp(file_mtime, tz=timezone.utc)
    #file_in_today_range = start_dt <= file_datetime <= end_dt
    if start_dt <= file_datetime <= end_dt:
        print(f"{file.split('/')[-1]}: start_dt={start_dt}, file_datetime={file_datetime}")
        files_to_process.append(file)


today_str=2026-11-08
PDF_SOURCES=['Morningstar Equity Research', 'EXCHANGE TRADED FUND REPORT CARD', 'Argus Recommendations', 'Argus A6 Quant']
mineru_api=<mineru.client.MinerU object at 0x7c12a83a29c0>
raw_folder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads
outfolder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed
z4f907d0azcd7aca12270d41c5898c6f25fb4a3094.pdf: start_dt=2026-11-08 00:00:00+00:00, file_datetime=2026-11-08 14:07:51.840528+00:00
z57907d0az69898d02347c4d8c9099171eb6239ba5.pdf: start_dt=2026-11-08 00:00:00+00:00, file_datetime=2026-11-08 14:07:18.207308+00:00
z57907d0az997fe88fd5414334be5fb46269cf4b81.pdf: start_dt=2026-11-08 00:00:00+00:00, file_datetime=2026-11-08 14:06:36.184534+00:00


In [2]:
# async def get_properties(file, md=None, pdf_sources=PDF_SOURCES) powered by AI
from openai import AsyncOpenAI
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this example.")

PROPERTIES = {
    "title": "",
    "source": "",
    "topic": "",
    "author": "",
    "published": "",
    "company_name": "",
    "symbol": "",
    "asof_date": "",
    "source_file": "",
    "created": today_str,
    "description": "",
} 

MODEL = "gpt-6-luna"
INSTRUCTIONS = (
    f"""
    Analyst reports in various formats will be provided as markdown input.
    Populate the properties dictionary '{PROPERTIES}' with information extracted from the markdown.  
    Put 'n/a' in any fields that are not available.
    Limit description to a concise summary of 200 words or less.
    Return the updated properties dictionary.
    """
)
async def get_properties(file, md=None, properties=PROPERTIES): #, pdf_sources=PDF_SOURCES):
    ''' gets title and source from markdown if available, else performs flash extraction and gets title and source '''
    import json

    if not md:
        result = mineru_api.flash_extract(file)
        md = result.markdown
        mode = "flash"
    else:
        mode = "md"

    client = AsyncOpenAI()
    response = await client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=md,
        #tools=tools,
        #tool_choice="auto",
    )
    for item in response.output:
        if hasattr(item, "content") and len(item.content) > 0:
            if hasattr(item.content[0], "text"):
                #break
                try:
                    properties = json.loads(item.content[0].text.replace("'",'"'))
                    #print(properties)
                except json.JSONDecodeError:
                    print("Failed to decode JSON")
                    properties = PROPERTIES

    properties = properties if 'properties' in locals() else {}
    properties["source_file"] = f'"[[{file.replace(f"{WIKI_FOLDER}/",'')}]]"'
    if "EXCHANGE TRADED FUND REPORT CARD" in md:
        properties["source"] = "ETrade_Morningstar"

    # string header for Obsidian properties 
    properties_str = "---\n"
    for p in properties:
        value = properties[p]
        properties_str += f"{p}: {value.replace(':','' )}\n"
    properties_str += "---\n"
    
    properties["output_filename"] = re.sub(r'[^a-zA-Z0-9\s]', '', properties["title"]).strip().replace(' ','_') + ".pdf.md"
    properties["properties_str"] = properties_str
    properties["md"] = md
    properties["mode"] = mode
    
    return properties


In [5]:
async def process_pdf_file(file, result=None):
    import json

    start_dt = datetime.now()
    now_str = start_dt.strftime("%Y-%m-%d %H-%M-%S")
    logging.info(f"{now_str}\tStarted: {file}")
    if not result:
        result = mineru_api.extract(  # https://pypi.org/project/mineru-open-sdk/
            file,
            ocr=True,
            language="en",
            file_params={},
        )
    md = result.markdown

    props = await get_properties(file=file, md=md)
    #print(json.dumps(props, indent=2))
    #print(props["properties_str"])

    # identify and write output file
    outfolder = f"{WIKI_FOLDER}/data/processed"
    outfile = f"{outfolder}/{props["source"].replace(' ','_')}/{today_str}/{props["output_filename"]}"
    props["output_file"] = outfile
    os.makedirs(os.path.dirname(outfile), exist_ok=True)
    with open(outfile, "w") as f:
        f.write(props["properties_str"] + md)

    # store images 
    images = result.images
    image_folder = f"{outfolder}/{props["source"].replace(' ','_')}/{today_str}/images"
    os.makedirs(image_folder, exist_ok=True)
    image_names = []
    for img in result.images:
        with open(f"{image_folder}/{img.name}", "wb") as f: 
            f.write(img.data)
        image_names.append(img.name)

    props["image_names"] = image_names
    import json
    #print(json.dumps(props, indent=2))
    with open(f"{WIKI_FOLDER}/data/processed/log_{today_str}.json", "a") as f:
        f.write(json.dumps(props, indent=2) + "\n")
        
    end_dt = datetime.now()
    now_str = end_dt.strftime("%Y-%m-%d %H-%M-%S")  
    duration = end_dt - start_dt
    logging.info(f"{now_str}\tFinished {outfile}\tProcessing time: {duration}")

    return props

#file = files_to_process[1]
for filepath in files_to_process:
    props = await process_pdf_file(filepath)
    #break
    #input(f"Processed file {filepath}. Press Enter to continue...")


    
    

INFO:root:2026-11-08 10-07-38	Started: /mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/raw/downloads/z4f907d0azcd7aca12270d41c5898c6f25fb4a3094.pdf
INFO:httpx:HTTP Request: POST https://mineru.net/api/v4/file-urls/batch "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: PUT https://mineru.oss-cn-shanghai.aliyuncs.com/api-upload/extract/2026-11-08/7ed61dff-112f-4805-a46a-dff7f82aabd1/7373ee2d-cb23-4621-a428-f74df38d7d6c.pdf?Expires=1794236858&OSSAccessKeyId=LTAI5t6sXp5gszd8yWZ85dAL&Signature=lEzWOnf699WDO%2BawLFH%2B4pBvVW4%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/7ed61dff-112f-4805-a46a-dff7f82aabd1 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/7ed61dff-112f-4805-a46a-dff7f82aabd1 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://mineru.net/api/v4/extract-results/batch/7ed61dff-112f-4805-a46a-dff7f82aabd1 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://miner

# Extract Tables from ETrade_Morningstar ETF Report Card .pdf into .json
## moved to [etrade_morningstar_extract](../backend/pdf_to_json_converter/etrade_morningstar_extract.ipynb)

Extracts .pdf tables into .json suitable for [import into DynamoDB](../backend/dyndb_load_json/dyndb_load_json.py) having structure based upon example output file [src/IDMO.msReportCard.json](IDMO.msReportCard.json).

(Tried extracting .json from the MinerU markdown, but results are inconsistent (skipped data), so using this separate PDF process.)

# initialize
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
import json

load_dotenv(override=True)

from datetime import datetime
today_str = datetime.today().strftime("%Y-%m-%d")
today_str = "2026-11-01"

WIKI_FOLDER="/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto"
md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{today_str}/"
json_folder = f"{md_folder}/json"
os.makedirs(json_folder, exist_ok=True)

print(f"md_folder={md_folder}")
print(f"json_folder={json_folder}")

files_to_process = glob.glob(md_folder + "/*.md")
files_to_process


def get_properties_from_markdown(md = None, md_filepath = None) -> dict:
    """Extract properties from markdown text if available."""
    if md is None and md_filepath is not None:
        with open(md_filepath, "r") as md_file:
            md = md_file.read()

    # extract Obsidian properties from the markdown front matter
    properties_str = md.split("---\n")[1]
    properties = {}
    for line in properties_str.split("\n"):
        if ":" in line:
            key, value = line.split(":", 1)
            properties[key.strip()] = value.strip()
    return properties

date_str = "2026-11-01"
md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{date_str}"
md_filepaths = glob.glob(md_folder + "/*.md")
for md_file in md_filepaths:
    print(md_file)
    properties = get_properties_from_markdown(md_filepath=md_file)
    print(properties)
    break


## Bedrock / LiteLLM configuration
REGION = os.environ.get("BEDROCK_REGION", "us-east-1")
os.environ["AWS_REGION_NAME"] = REGION  # LiteLLM's preferred variable
os.environ["AWS_REGION"] = REGION  # Boto3 standard
os.environ["AWS_DEFAULT_REGION"] = REGION  # Fallback

BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.amazon.nova-pro-v1:0")
MODEL = f"bedrock/{BEDROCK_MODEL_ID}"

schema_folder = os.getenv("SCHEMA_FOLDER", "./")
SCHEMA_EXAMPLE_PATH = f"{schema_folder}/IDMO.msReportCard.json"

pdf_folder = os.getenv("PDF_FOLDER", "./")
md_folder = os.getenv("MARKDOWN_FOLDER", "./")

INSTRUCTIONS = """You convert E*TRADE Exchange Traded Fund Report Card text into structured JSON.

You will be given the raw text extracted from a *.msReportCard.pdf file. Produce a JSON object
that follows EXACTLY the same keys, nesting, and structure as this example (the example's values
belong to a different fund and must not be reused):

{example_json}

Rules:
- Use the ticker symbol found in the source text for "symbol".
- "asof_dt" is the "Average Annual Return & Tax Analysis as of" date in MM/DD/YY format
  (this is usually a month-end date, e.g. 07/31/26), NOT the "Report created on" date and NOT the
  "Close on" date.
- Morningstar star ratings (e.g. "morningstar_star_rating" and "rating" fields) must use the
  literal capital letter "H" repeated once per star (e.g. 5 stars = "HHHHH"), not the "\u2605" star
  glyph.
- Numbers must be plain JSON numbers (no "%", "$", or "," characters); percentages are represented
  as their numeric value (e.g. 23.91 for 23.91%).
- If a value is not present in the source text, use null.
- Respond with ONLY the JSON object, no markdown fences, no commentary.
"""

from pypdf import PdfReader

def extract_pdf_text(pdf_path: Path) -> str:
    """Extract concatenated text from every page of a PDF."""
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def _parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model response: {text[:200]!r}")
    return json.loads(cleaned[start : end + 1])

async def convert_pdf_text_to_json(pdf_text: str) -> dict:
    """Send extracted PDF text to Bedrock (via LitellmModel) and return parsed JSON."""
    from agents import Agent, ModelSettings, Runner, trace
    from agents.extensions.models.litellm_model import LitellmModel

    with open(SCHEMA_EXAMPLE_PATH) as f:
        example_json = f.read()

    agent = Agent(
        name="msReportCard Converter",
        instructions=INSTRUCTIONS.format(example_json=example_json),
        model=LitellmModel(model=MODEL),
        model_settings=ModelSettings(max_tokens=4096),
    )
    with trace("msrportcard_converter"):
        result = await Runner.run(agent, input=pdf_text)
    
    return _parse_json_response(result.final_output)


# process *.pdf files and upload corresponding JSON to S3
import boto3

date_str = "2026-11-01"
md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{date_str}"
md_filepaths = glob.glob(md_folder + "/*.md")
md_filepaths

json_folder = os.getenv("JSON_FOLDER", f"{md_folder}/json")

for md_filepath in md_filepaths:
    print(md_filepath)
    properties = get_properties_from_markdown(md_filepath=md_filepath)
    print(properties)

    # get .pdf path from .md properties
    source_file = properties.get("source_file").replace('"[[', '').replace(']]"', '')
    pdf_path = f"{WIKI_FOLDER}/{source_file}"
    print(pdf_path)

    pdf_text = extract_pdf_text(pdf_path)
    json_data = await convert_pdf_text_to_json(pdf_text)
    json_data["properties"] = properties

    json_path = f"{json_folder}/{properties["symbol"]}.msReportCard.json"
    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=4)
    print(f"Wrote {json_path}")

    # upload .json to S3 to be processed by ObjectCreated-initiated Lambda ramlytics_dyndb_load_json
    s3_bucket = os.getenv("S3_BUCKET", "fh-danelfin-289755104220")
    s3_inbound_folder = "inbound/market_data"
    s3_client = boto3.client("s3")
    s3_key = f"{s3_inbound_folder}/{properties['symbol']}.msReportCard.json"
    s3_client.upload_file(str(json_path), s3_bucket, s3_key)
    print(f"Uploaded to s3://{s3_bucket}/{s3_key}")
